# **Fruits-360 Classification**

In [ ]:
# ── CELL 1: Upload Kaggle API Key ────────────────────────────
# Heading: Step 1: Upload Kaggle API key

from google.colab import files
files.upload()   # Upload your kaggle.json file here

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"hema34nt","key":"1c883966e547eaab3b052c3b69736cc1"}'}

In [ ]:
# ── CELL 2: Setup Kaggle & Download Dataset ──────────────────
# Heading: Step 2: Setup Kaggle & Download Dataset

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d moltean/fruits

Dataset URL: https://www.kaggle.com/datasets/moltean/fruits
License(s): CC-BY-SA-4.0
100% 6.24G/6.24G [00:43<00:00, 155MB/s]



In [ ]:
# ── CELL 3: Check Downloaded Files ──────────────────────────
# Heading: Step 3: Check Downloaded Files

!ls


fruits.zip  kaggle.json  sample_data


In [ ]:

# ── CELL 4: Unzip and Organize ──────────────────────────────
# Heading: Step 4: Unzip and Organize

!unzip -q fruits.zip -d fruits_data

In [ ]:
# ── CELL 5: Check Folder Structure ──────────────────────────
# Heading: Step 5: Check Folder Structure

# Let's see exactly what is inside the unzipped folder to fix the paths
!ls -R fruits_data/fruits-360_100x100 | head -n 20

fruits_data/fruits-360_100x100:
fruits-360

fruits_data/fruits-360_100x100/fruits-360:
LICENSE
README.md
Test
Training

fruits_data/fruits-360_100x100/fruits-360/Test:
Almonds 1
Apple 10
Apple 11
Apple 12
Apple 13
Apple 14
Apple 17
Apple 18
Apple 19
Apple 20


In [ ]:
# ── CELL 6: Import Libraries ─────────────────────────────────
# Heading: Step 6: Import Libraries

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import os

In [ ]:
# ── CELL 7: Image Settings ───────────────────────────────────
# Heading: Step 7: Image Settings

IMG_SIZE = (100, 100)
BATCH_SIZE = 32
train_dir = "fruits_data/fruits-360/Training"
val_dir = "fruits_data/fruits-360/Test"

In [ ]:
# ── CELL 8: Image Generators ─────────────────────────────────
# Heading: Step 8: Image Generators

# Updated paths based on detailed ls -R output from Cell 5
train_dir = "fruits_data/fruits-360_100x100/fruits-360/Training"
val_dir = "fruits_data/fruits-360_100x100/fruits-360/Test"

train_gen = ImageDataGenerator(rescale=1./255)
val_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_data = val_gen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

Found 137221 images belonging to 260 classes.
Found 45724 images belonging to 260 classes.


In [ ]:
# ── CELL 9: Build CNN Model ──────────────────────────────────
# Heading: Step 9: Build CNN Model

model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(100, 100, 3)),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(train_data.num_classes, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# ── CELL 10: Compile the Model ───────────────────────────────
# Heading: Step 10: Compile the Model

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# ── CELL 11: Train the Model ─────────────────────────────────
# Heading: Step 11: Train the Model

history = model.fit(
    train_data,
    epochs=5,
    validation_data=val_data
)

Epoch 1/5
4289/4289 ━━━━━━━━━━━━━━━━━━━━ 2717s 633ms/step - accuracy: 0.8273 - loss: 0.6289 - val_accuracy: 0.9422 - val_loss: 0.5811
Epoch 2/5
4289/4289 ━━━━━━━━━━━━━━━━━━━━ 2660s 620ms/step - accuracy: 0.9578 - loss: 0.1210 - val_accuracy: 0.9460 - val_loss: 0.6609
Epoch 3/5
3228/4289 ━━━━━━━━━━━━━━━━━━━━ 9:59 565ms/step - accuracy: 0.9686 - loss: 0.0889

In [ ]:
# ── CELL 12: Plot Accuracy and Loss ──────────────────────────
# Heading: Step 12: Plot Accuracy and Loss

plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title("Model Accuracy")
plt.show()

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title("Model Loss")
plt.show()

In [ ]:
# ── CELL 13: Upload a New Test Image ─────────────────────────
# Heading: Step 13: Upload a New Test Image

from google.colab import files
uploaded = files.upload()
# This will prompt you to upload an image (e.g., banana.jpg)
# Make sure it's a clear image of a fruit.

In [ ]:
# ── CELL 14: Load and Preprocess the Image ───────────────────
# Heading: Step 14: Load and Preprocess the Image

import numpy as np
from tensorflow.keras.utils import load_img, img_to_array

# Replace 'banana.jpg' with your image name
image_path = list(uploaded.keys())[0]   # Automatically gets the uploaded image
img = load_img(image_path, target_size=IMG_SIZE)
plt.imshow(img)
plt.axis('off')
plt.title("Test Image")
plt.show()

# Preprocess image
img_array = img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)   # Add batch dimension

In [ ]:
# ── CELL 15: Make Prediction ─────────────────────────────────
# Heading: Step 15: Make Prediction

pred = model.predict(img_array)
pred_class_index = np.argmax(pred)
pred_class_label = list(train_data.class_indices.keys())[pred_class_index]

print("Predicted class:", pred_class_label)
# This prints the predicted fruit class (e.g., "Banana", "Apple", etc.)

In [ ]:
# ── CELL 16 (Optional): See All Class Labels ────────────────
# Heading: (Optional) See All Class Labels and Indices

print("Class indices:", train_data.class_indices)
